In [13]:
import cv2
import json

# === Set image path ===
IMAGE_PATH = "test_image.jpeg"  # Change this to your actual image file
OUTPUT_FILE = "lshape_image.json"

# Store up to 3 clicked points
points = []

# Mouse callback to collect L-shape points
def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        points.append((x, y))

# Load the image
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

cv2.namedWindow("Define L-Shape ROI", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Define L-Shape ROI", 1000, 800)
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()

    # Draw selected points
    for point in points:
        cv2.circle(display_frame, point, 5, (0, 0, 255), -1)

    # Draw L-shape lines
    if len(points) >= 2:
        cv2.line(display_frame, points[0], points[1], (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, points[1], points[2], (0, 255, 0), 2)

    # Instructions
    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape to {OUTPUT_FILE}")
    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()


[✅] Saved L-shape to lshape_image.json


In [ ]:
import cv2
import json

# === Set image path ===
IMAGE_PATH = "plain.jpeg"  # Change this to your actual image file
OUTPUT_FILE = "lshape_image.json"
q
# Store up to 3 clicked points
points = []

# Mouse callback to collect L-shape points
def draw_lshape(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        points.append((x, y))

# Load the image
frame = cv2.imread(IMAGE_PATH)
if frame is None:
    print(f"❌ Failed to load image: {IMAGE_PATH}")
    exit()

cv2.namedWindow("Define L-Shape ROI")
cv2.setMouseCallback("Define L-Shape ROI", draw_lshape)

while True:
    display_frame = frame.copy()

    # Draw selected points
    for point in points:
        cv2.circle(display_frame, point, 5, (0, 0, 255), -1)

    # Draw L-shape lines
    if len(points) >= 2:
        cv2.line(display_frame, points[0], points[1], (0, 255, 0), 2)
    if len(points) == 3:
        cv2.line(display_frame, points[1], points[2], (0, 255, 0), 2)

    # Instructions
    cv2.putText(display_frame, "Click 3 points for L-shape. 's'=Save, 'r'=Reset, 'q'=Quit",
                (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow("Define L-Shape ROI", display_frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s') and len(points) == 3:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(points, f)
        print(f"[✅] Saved L-shape to {OUTPUT_FILE}")
    elif key == ord('r'):
        points = []
    elif key == ord('q'):
        break

cv2.destroyAllWindows()


[✅] Saved L-shape to lshape_image.json


In [15]:
import cv2
import json
import numpy as np
from ultralytics import YOLO  # pip install ultralytics

# === Load L-shape points from JSON ===
with open("lshape_image.json", "r") as f:
    l_points = json.load(f)

A, B, C = [tuple(map(int, pt)) for pt in l_points]  # Ensure points are tuples of int

# === Create L-shape lines ===
line1 = (np.array(A), np.array(B))
line2 = (np.array(B), np.array(C))

# === Load YOLOv8 model ===
model_path = "yolo11n.pt"  # Replace with your trained model path
model = YOLO(model_path)
print(f"✅ YOLO model loaded from: {model_path}")

# === Load test image ===
image_path = "test_image.jpeg"  # Change to your actual test image
frame = cv2.imread(image_path)
if frame is None:
    print(f"❌ Failed to load image: {image_path}")
    exit()

# === Run YOLOv8 detection ===
results = model(frame)

# === Draw YOLO detections ===
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    label = model.names[cls_id]
    conf = box.conf[0]

    if "truck" in label.lower():  # You can change to 'person' or others
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# === Draw L-shape on image ===
cv2.line(frame, A, B, (0, 255, 255), 2)
cv2.line(frame, B, C, (0, 255, 255), 2)

cv2.circle(frame, A, 5, (0, 0, 255), -1)
cv2.circle(frame, B, 5, (0, 255, 0), -1)
cv2.circle(frame, C, 5, (255, 0, 0), -1)

cv2.putText(frame, "A", A, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
cv2.putText(frame, "B", B, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
cv2.putText(frame, "C", C, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

# === Show image ===
cv2.namedWindow("Test Detection with L-Shape", cv2.WINDOW_NORMAL)
#cv2.resizeWindow("Test Detection with L-Shape", 1000, 800)
cv2.imshow("Test Detection with L-Shape", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()


✅ YOLO model loaded from: yolo11n.pt

0: 640x480 1 truck, 2 chairs, 1 couch, 148.2ms
Speed: 4.0ms preprocess, 148.2ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)
